In [3]:
from PandaTableVoxelClutterGenerator import PandaTableVoxelClutterGenerator
import robotic as ry


def main():
    generator = PandaTableVoxelClutterGenerator(
        base_scene_file=ry.raiPath("../rai-robotModels/scenarios/pandaSingle.g"),  # Path to the base Panda scene .g file. Usually ry.raiPath("..."). Must point to an existing scene file.
        voxel_dir="../voxel_generation/data",                                      # Folder containing voxel .g files to sample from. Any valid directory path string.
        output_dir="./generated_envs",                                             # Folder where the generated environment file will be saved. Any valid directory path string.
        table_frame_name="table",                                                  # Name of the table frame inside the base scene. Must match a frame name in the loaded scene, e.g. "table".
        gap=0.04,                                                                  # Extra XY margin used during placement to keep objects away from borders / each other. Any nonnegative float; larger = more spacing.
        spawn_height=0.41,                                                         # Initial extra Z height above the table when dropping voxels into physics. Positive float; larger = dropped from higher up.
        seed=42,                                                                   # Random seed for reproducible target choice, quarter choice, rotations, and spawn positions. Any integer.
        per_cube_mass=0.2,                                                         # Mass assigned to each cube piece of each voxel object. Positive float; larger = heavier cubes.
        table_shape_size=(1.6, 1.6, 0.08, 0.02),                                   # Table shape parameters passed to ssBox. Typically (size_x, size_y, size_z, rounding_radius).
        panda_base_relative_pos=(0.0, 0.0, 0.05),                                  # Relative position assigned to frame "l_panda_base". Typically a 3-tuple/list (x, y, z).
        target_alpha=0.9,                                                          # Transparency for the goal_pose_ object only. Usually in [0, 1]: 0 = fully transparent, 1 = fully opaque.
        target_center_jitter_ratio=0.05,                                           # How much the goal_pose_ location is jittered around the target quarter center. Usually in [0, 1]; smaller = closer to exact center.
        clutter_mode="low_clutter",                                                # Clutter placement strategy. Allowed values: "random", "low_clutter", "high_clutter".
        placement_candidate_count=128,                                             # Number of sampled candidate placements considered in low/high clutter modes. Positive integer; larger = slower but often better placement quality.
        hardnessOfTargetObject=0.0,                                                # Controls when the goal_ clutter object is inserted among clutter drops. Must be in [0, 1]: 0.0 = as late as possible, 1.0 = as early as possible. (0 is easier as the goal object is on top, 1 is the hardest)
    )

    C, summary = generator.create_environment_with_refill(
        num_voxels=8,                 # Desired final clutter count INCLUDING exactly one goal_ object, but EXCLUDING the extra goal_pose_ object. Positive integer.
        sim_seconds=10.0,             # How long to run physics after each spawn/refill phase. Positive float; larger = more settling time.
        sim_dt=0.01,                  # Physics simulation time step. Positive float; smaller = finer simulation but slower.
        max_refill_rounds=20,         # Maximum number of refill/simulate/remove rounds for reaching the target clutter count. Positive integer.
        xy_margin=0.02,               # Margin inside table XY bounds when deciding whether an object is still considered "on the table". Nonnegative float.
        z_tolerance=0.15,             # Allowed drop below table-top height when deciding whether an object survived on the table. Nonnegative float.
        batch_spawn_count=5,          # Maximum number of clutter objects to attempt to spawn in one refill batch. Positive integer.
        max_target_spawn_attempts=40, # Maximum number of rescue attempts for goal_ and goal_pose_ spawning. Positive integer.
    )

    print("\n--- Scene summary ---")
    print(f"Clutter mode                 : {summary['clutter_mode']}")
    print(f"Final on table               : {summary['final_on_table']}")
    print(f"Final off table              : {summary['final_off_table']}")
    print(f"Refill rounds                : {summary['rounds']}")
    print(f"hardnessOfTargetObject       : {summary['hardnessOfTargetObject']}")
    print(f"goal_ insert index           : {summary['goal_clutter_insert_index']}")

    print("\n--- Target voxel info ---")
    print(f"Target voxel file            : {summary['target_voxel_file']}")
    print(f"Original voxel basename      : {summary['target_original_voxel_basename']}")
    print(f"goal_ voxel basename         : {summary['goal_clutter_voxel_basename']}")
    print(f"goal_pose_ voxel basename    : {summary['goal_pose_voxel_basename']}")
    print(f"Target quarter               : {summary['target_quarter_mode']}")
    print(f"goal_ object prefix          : {summary['goal_clutter_object_prefix']}")
    print(f"goal_pose_ object prefix     : {summary['goal_pose_object_prefix']}")
    print(f"goal_pose alpha              : {summary['target_alpha']}")

    alive_objects = [obj for obj in summary["objects"] if obj["alive"]]
    print(f"\nAlive spawned objects        : {len(alive_objects)}")
    print("Alive voxel basenames        :", [obj["basename"] for obj in alive_objects])

    goal_alive = [obj for obj in alive_objects if obj.get("role") == "goal"]
    goal_pose_alive = [obj for obj in alive_objects if obj.get("role") == "goal_pose"]
    normal_clutter_alive = [obj for obj in alive_objects if obj.get("role") == "normal"]
    all_clutter_alive = [obj for obj in alive_objects if obj.get("counts_as_clutter", False)]

    print(f"\nAlive goal_ objects          : {len(goal_alive)}")
    print(f"Alive goal_pose_ objects     : {len(goal_pose_alive)}")
    print(f"Alive normal clutter objects : {len(normal_clutter_alive)}")
    print(f"Alive total clutter objects  : {len(all_clutter_alive)}")

    if goal_alive:
        tgt = goal_alive[0]
        print("\n--- Alive goal_ object details ---")
        print(f"Prefix                       : {tgt['prefix']}")
        print(f"Basename                     : {tgt['basename']}")
        print(f"Original basename            : {tgt['original_basename']}")
        print(f"Role                         : {tgt['role']}")
        print(f"Spawn XY                     : {tgt['spawn_xy']}")
        print(f"Spawn Z                      : {tgt['spawn_z']}")
        print(f"Theta deg                    : {tgt['theta_deg']}")
        print(f"Region modes                 : {tgt['region_modes']}")
        print(f"Counts as clutter            : {tgt['counts_as_clutter']}")
        print(f"Clutter mode                 : {tgt.get('clutter_mode')}")

    if goal_pose_alive:
        tgt_pose = goal_pose_alive[0]
        print("\n--- Alive goal_pose_ object details ---")
        print(f"Prefix                       : {tgt_pose['prefix']}")
        print(f"Basename                     : {tgt_pose['basename']}")
        print(f"Original basename            : {tgt_pose['original_basename']}")
        print(f"Role                         : {tgt_pose['role']}")
        print(f"Spawn XY                     : {tgt_pose['spawn_xy']}")
        print(f"Spawn Z                      : {tgt_pose['spawn_z']}")
        print(f"Theta deg                    : {tgt_pose['theta_deg']}")
        print(f"Region modes                 : {tgt_pose['region_modes']}")
        print(f"Alpha                        : {tgt_pose['target_alpha']}")
        print(f"Counts as clutter            : {tgt_pose['counts_as_clutter']}")

    if normal_clutter_alive:
        print("\n--- Alive normal clutter object details ---")
        for i, obj in enumerate(normal_clutter_alive, start=1):
            print(f"\nNormal clutter #{i}")
            print(f"  Prefix                     : {obj['prefix']}")
            print(f"  Basename                   : {obj['basename']}")
            print(f"  Original basename          : {obj['original_basename']}")
            print(f"  Role                       : {obj['role']}")
            print(f"  Spawn XY                   : {obj['spawn_xy']}")
            print(f"  Spawn Z                    : {obj['spawn_z']}")
            print(f"  Theta deg                  : {obj['theta_deg']}")
            print(f"  Region modes               : {obj['region_modes']}")
            print(f"  Clutter mode               : {obj.get('clutter_mode')}")
            print(f"  Counts as clutter          : {obj['counts_as_clutter']}")

    saved_path = generator.save_environment(
        C,
        file_name="new_environment.g",  # Output file name for the saved generated scene. Any valid file name string ending with .g is recommended.
    )
    print("\nSaved to:", saved_path)

    # C.view(True)  # Opens the rai visualizer if uncommented.


if __name__ == "__main__":
    main()

Chosen target voxel: 107.g
Chosen target quarter: back_right
Clutter mode: low_clutter
hardnessOfTargetObject: 0.0
goal_ clutter insert index: 7
Initially spawned clutter voxels: 5 / 5

=== Clutter simulation round 1 ===
Clutter on table: 5 | Off table: 0 | Base clutter target: 8 | Dynamic clutter target: 9 | goal_ on table: 0
Trying to respawn up to 4 clutter voxel(s)... (goal present on table? no)
Respawned 4 clutter voxel(s).

=== Clutter simulation round 2 ===
Clutter on table: 8 | Off table: 1 | Base clutter target: 8 | Dynamic clutter target: 8 | goal_ on table: 1
Desired clutter count is on the table, including a surviving goal_ target.
Removing 1 final off-table clutter voxel(s).

=== Final rescue check for goal_ ===

=== Final goal_ clutter survival check ===
Target voxel file         : ../voxel_generation/data/107.g
Target quarter            : back_right
Allowed clutter regions   : ['front_left', 'front_right', 'back_left']
On-table bounds (margin)  : x[-0.7800, 0.7800] y[-0.

In [4]:
# The generated environment is saved to generated_envs
C = ry.Config()
C.addFile("generated_envs/new_environment.g") 

C.view()


0

In [9]:
C.getFrameNames()

['world',
 'table',
 'l_panda_base',
 'l_panda_link0',
 'l_panda_joint1_origin',
 'l_panda_joint1',
 'l_panda_joint2_origin',
 'l_panda_joint2',
 'l_panda_joint3_origin',
 'l_panda_joint3',
 'l_panda_joint4_origin',
 'l_panda_joint4',
 'l_panda_joint5_origin',
 'l_panda_joint5',
 'l_panda_joint6_origin',
 'l_panda_joint6',
 'l_panda_joint7_origin',
 'l_panda_joint7',
 'l_panda_joint8_origin',
 'l_panda_joint8',
 'l_panda_hand_joint_origin',
 'l_panda_hand_joint',
 'l_panda_finger_joint1_origin',
 'l_panda_finger_joint2_origin',
 'l_panda_finger_joint1',
 'l_panda_finger_joint2',
 'l_panda_rightfinger_0',
 'l_panda_coll0',
 'l_panda_coll1',
 'l_panda_coll3',
 'l_panda_coll5',
 'l_panda_coll2',
 'l_panda_coll4',
 'l_panda_coll6',
 'l_panda_coll7',
 'l_gripper',
 'l_palm',
 'l_finger1',
 'l_finger2',
 'cameraTop',
 'cameraWrist',
 'panda_collCameraWrist',
 'obj0_base',
 'obj0_cube0',
 'obj0_joint1',
 'obj0_cube1',
 'obj0_joint2',
 'obj0_cube2',
 'obj0_joint3',
 'obj0_cube3',
 'obj0_joint4